# 13 · Adataugmentáció – Osztályegyensúly Helyreállítása

**Probléma:** A train split erősen kiegyensúlyozatlan (No hand=10, G=15 vs. B=39 kép).  
**Megoldás:** Offline augmentáció – valós képekből mesterséges változatok generálása:  
forgatás · fényerő · kontraszt · Gaussian-zaj · blur · perspektíva-warp · vágás+resize

**Kimenet:**
- `data/augmented/{osztály}/` – augmentált JPG képek
- `data/split_manifest_aug.csv` – kibővített manifest
- `data/features/X_*.npy` – újragenerált feature fájlok (augmentált adattal)

> **Csak a TRAIN split kap augmentált képeket** – val/test érintetlen marad.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from IPython.display import display, HTML
import seaborn as sns

from src.config import PATHS
from src.augmentor import (
    augment_one, run_augmentation,
    ROTATE_MAX, BRIGHT_LO, BRIGHT_HI,
    CONTR_LO, CONTR_HI, NOISE_LO, NOISE_HI,
)

MANIFEST   = PATHS["manifest"]
FEAT_DIR   = PATHS["data"] / "features"
AUG_ROOT   = PATHS["data"] / "augmented"
SEED       = 42
rng        = np.random.default_rng(SEED)

print(f"Manifest   : {MANIFEST}")
print(f"Aug kimenet: {AUG_ROOT}")
print(f"Features   : {FEAT_DIR}")

## 1 · Jelenlegi Eloszlás

In [ ]:
df       = pd.read_csv(MANIFEST)
classes  = sorted(df["class"].unique().tolist())
train_df = df[df["split"] == "train"]

train_counts = train_df["class"].value_counts().reindex(classes).fillna(0).astype(int)

# ── Bar chart: train eloszlás + TARGET vonal ───────────────────────────────────
TARGET = 50

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bal: teljes eloszlás split szerint
pivot = df.groupby(["class","split"]).size().unstack(fill_value=0).reindex(classes)
colors = {"train":"#4C72B0", "val":"#DD8452", "test":"#55A868"}
bottom = np.zeros(len(classes))
for sp in ["train","val","test"]:
    vals = pivot.get(sp, pd.Series(0, index=classes))
    bars = axes[0].bar(classes, vals, bottom=bottom,
                       label=sp, color=colors[sp], edgecolor="white")
    bottom += vals.values
axes[0].set_title("Teljes adathalmaz (split szerint)", fontweight="bold")
axes[0].set_ylabel("Darabszám")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Jobb: train-only + target vonal
bar_colors = ["#e74c3c" if v < TARGET else "#4C72B0" for v in train_counts.values]
bars = axes[1].bar(classes, train_counts.values, color=bar_colors, edgecolor="white")
axes[1].axhline(TARGET, color="black", linestyle="--", linewidth=1.5,
                label=f"Cél: {TARGET}/osztály")
axes[1].bar_label(bars, padding=2, fontsize=9)
axes[1].set_title("Train split (piros = cél alatt)", fontweight="bold")
axes[1].set_ylabel("Darabszám")
axes[1].set_ylim(0, max(train_counts.max(), TARGET) * 1.2)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Osztályeloszlás augmentálás ELŐTT", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Táblázat
print(f"\n{'Osztály':10s}  {'Train':>6s}  {'Val':>5s}  {'Test':>5s}  {'Hiány':>6s}")
print("─" * 42)
for cls in classes:
    tr = (df["split"]=="train") & (df["class"]==cls)
    va = (df["split"]=="val")   & (df["class"]==cls)
    te = (df["split"]=="test")  & (df["class"]==cls)
    need = max(0, TARGET - tr.sum())
    flag = " ← augmentálni kell" if need > 0 else ""
    print(f"  {cls:8s}  {tr.sum():6d}  {va.sum():5d}  {te.sum():5d}  {need:6d}{flag}")

## 2 · Augmentáció Előnézet

Egy forráskép 8 különböző augmentált változata — így néz ki, amit a pipeline majd megkap.

In [ ]:
# Vegyük az egyik leghiányosabb osztály első képét
def _weakest_class_sample():
    counts = train_df["class"].value_counts()
    weakest = counts.idxmin()
    row = train_df[train_df["class"] == weakest].iloc[0]
    return weakest, Path(row["path"])

weak_cls, sample_path = _weakest_class_sample()
original = cv2.imread(str(sample_path))
original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 5, figure=fig, hspace=0.3, wspace=0.08)

# Eredeti
ax0 = fig.add_subplot(gs[:, 0])
ax0.imshow(original_rgb)
ax0.set_title(f"Eredeti\n({weak_cls})", fontweight="bold", color="#2c3e50")
ax0.axis("off")

# 8 augmentált változat
rng_prev = np.random.default_rng(SEED + 1)
for idx in range(8):
    aug_bgr = augment_one(original, rng_prev)
    aug_rgb = cv2.cvtColor(aug_bgr, cv2.COLOR_BGR2RGB)
    row_i, col_i = divmod(idx, 4)
    ax = fig.add_subplot(gs[row_i, col_i + 1])
    ax.imshow(aug_rgb)
    ax.set_title(f"aug {idx+1}", fontsize=9)
    ax.axis("off")

plt.suptitle(
    f"Augmentációs előnézet – '{weak_cls}' osztály  "
    f"(forgatás±{ROTATE_MAX}°, fényerő, kontraszt, zaj, blur, perspektíva, vágás)",
    fontsize=12, fontweight="bold",
)
plt.show()

## 3 · Augmentáció Futtatása

Az alábbi cella **blokkol** amíg minden augmentált kép el nem készül.  
Időigény: tipikusan < 1 perc (csak képmanipuláció, pipeline nem fut).

In [ ]:
display(HTML(
    "<div style='border:2px solid #f39c12; border-radius:6px; padding:10px; "
    "background:#fef9e7; font-family:monospace'>"
    f"<b>⏳ Augmentáció indul – cél: {TARGET} kép/osztály (train)</b><br>"
    "Csak a cél alatt lévő osztályok kapnak augmentált képeket."
    "</div>"
))

full_df = run_augmentation(
    manifest_path=MANIFEST,
    output_root=AUG_ROOT,
    target_per_class=TARGET,
    seed=SEED,
)

n_aug = (full_df.get("source", "original") == "augmented").sum()
display(HTML(
    "<div style='border:2px solid #27ae60; border-radius:6px; padding:10px; "
    "background:#eafaf1; font-family:monospace; margin-top:8px'>"
    f"<b>✅ Augmentáció kész!</b>  {n_aug} új kép generálva."
    "</div>"
))

## 4 · Eloszlás Augmentálás Után

In [ ]:
train_aug = full_df[full_df["split"] == "train"]
train_new_counts = train_aug["class"].value_counts().reindex(classes).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bal: before
train_counts_arr = train_counts.reindex(classes).values
before_colors = ["#e74c3c" if v < TARGET else "#4C72B0" for v in train_counts_arr]
bars_b = axes[0].bar(classes, train_counts_arr, color=before_colors, edgecolor="white")
axes[0].axhline(TARGET, color="black", linestyle="--", linewidth=1.5, label=f"Cél: {TARGET}")
axes[0].bar_label(bars_b, padding=2, fontsize=9)
axes[0].set_title("ELŐTTE (eredeti)", fontweight="bold")
axes[0].set_ylabel("Train darabszám")
axes[0].set_ylim(0, max(train_new_counts.max(), TARGET) * 1.2)
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

# Jobb: after – eredeti (kék) + augmentált (narancs) réteg
original_in_train = (
    full_df[(full_df["split"]=="train") & (full_df.get("source","original")!="augmented")]
    ["class"].value_counts().reindex(classes).fillna(0).astype(int)
)
aug_in_train = (
    full_df[(full_df["split"]=="train") & (full_df.get("source","original")=="augmented")]
    ["class"].value_counts().reindex(classes).fillna(0).astype(int)
)

bars_o = axes[1].bar(classes, original_in_train.values,
                     label="Eredeti", color="#4C72B0", edgecolor="white")
axes[1].bar(
    classes, aug_in_train.values, bottom=original_in_train.values,
    label="Augmentált", color="#DD8452", edgecolor="white", alpha=0.85,
)
axes[1].axhline(TARGET, color="black", linestyle="--", linewidth=1.5,
                label=f"Cél: {TARGET}")
for i, (o, a) in enumerate(zip(original_in_train.values, aug_in_train.values)):
    axes[1].text(i, o + a + 0.5, str(o + a), ha="center", va="bottom", fontsize=9)
axes[1].set_title("UTÁNA (eredeti + augmentált)", fontweight="bold")
axes[1].set_ylabel("Train darabszám")
axes[1].set_ylim(0, max(train_new_counts.max(), TARGET) * 1.2)
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Train eloszlás augmentálás ELŐTT vs. UTÁN",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n{'Osztály':10s}  {'Eredeti':>8s}  {'Aug.':>6s}  {'Összesen':>8s}")
print("─" * 40)
for cls in classes:
    o = int(original_in_train.get(cls, 0))
    a = int(aug_in_train.get(cls, 0))
    print(f"  {cls:8s}  {o:8d}  {a:6d}  {o+a:8d}")

## 5 · Augmentált Képek Galériája

Mentett augmentált képek véletlenszerű mintája osztályonként.

In [ ]:
aug_only = full_df[full_df.get("source", pd.Series("original", index=full_df.index)) == "augmented"]

fig, axes = plt.subplots(len(classes), 5, figsize=(14, len(classes) * 2.4))
for row_i, cls in enumerate(classes):
    cls_aug = aug_only[aug_only["class"] == cls]
    sample  = cls_aug.sample(min(5, len(cls_aug)), random_state=SEED) if len(cls_aug) > 0 else pd.DataFrame()
    for col_i in range(5):
        ax = axes[row_i, col_i]
        if col_i < len(sample):
            img_path = Path(sample.iloc[col_i]["path"])
            img      = cv2.imread(str(img_path))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if col_i == 0:
            ax.set_ylabel(cls, rotation=0, labelpad=45, fontsize=11,
                          fontweight="bold", va="center")
        ax.axis("off")

plt.suptitle("Augmentált képek mintái (5 per osztály)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6 · Feature Export Újrafuttatása

Az augmentált manifest alapján újrageneráljuk az npy fájlokat.  
**Időigény: ~8-15 perc** (eredeti 297 + augmentált képek × V14 pipeline).

A cella blokkol és **tqdm** bar-ral mutatja a haladást.

In [ ]:
aug_manifest_path = PATHS["data"] / "split_manifest_aug.csv"

if not aug_manifest_path.exists():
    display(HTML("<b style='color:red'>❌ Manifest nem található – futtasd előbb a 3. cellát!</b>"))
else:
    n_total = len(pd.read_csv(aug_manifest_path))
    display(HTML(
        f"<div style='border:2px solid #f39c12; border-radius:6px; padding:10px; "
        f"background:#fef9e7; font-family:monospace'>"
        f"<b>⏳ Feature export indul – {n_total} kép × V14 pipeline</b><br>"
        f"Eredmény: <code>data/features/X_*.npy</code> (felülírja a régit)"
        f"</div>"
    ))

    from src.dataset_generator import export_dataset
    export_dataset(
        manifest_path=aug_manifest_path,
        output_dir=FEAT_DIR,
        verbose=True,
    )

    import numpy as np
    rows = ""
    for fname in ["X_basic.npy","X_inlay.npy","X_images.npy","y.npy"]:
        a = np.load(FEAT_DIR / fname)
        rows += f"<tr><td>{fname}</td><td>{a.shape}</td><td>{a.dtype}</td></tr>"

    display(HTML(
        "<div style='border:2px solid #27ae60; border-radius:6px; padding:10px; "
        "background:#eafaf1; font-family:monospace; margin-top:8px'>"
        "<b>✅ Export kész!</b><br><br>"
        "<table style='border-collapse:collapse'>"
        "<tr><th style='padding:2px 12px'>Fájl</th>"
        "<th style='padding:2px 12px'>Shape</th>"
        "<th style='padding:2px 12px'>Dtype</th></tr>"
        + rows + "</table></div>"
    ))

## 7 · Végső Eloszlás Ellenőrzése (npy alapján)

In [ ]:
import numpy as np

y_new    = np.load(FEAT_DIR / "y.npy")
spl_new  = np.load(FEAT_DIR / "splits.npy", allow_pickle=True)
cls_new  = list(np.load(FEAT_DIR / "class_names.npy", allow_pickle=True))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(cls_new))
w = 0.28
for offset, split, color in [
    (-w, "train", "#4C72B0"),
    ( 0, "val",   "#DD8452"),
    (+w, "test",  "#55A868"),
]:
    vals = [(((y_new==i) & (spl_new==split)).sum()) for i in range(len(cls_new))]
    bars = ax.bar(x + offset, vals, w, label=split, color=color, edgecolor="white")
    ax.bar_label(bars, padding=1, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(cls_new)
ax.axhline(TARGET, color="black", linestyle="--", linewidth=1.2,
           label=f"Cél: {TARGET}")
ax.set_title("Npy-alapú eloszlás augmentálás után",
             fontweight="bold", fontsize=12)
ax.set_ylabel("Darabszám")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nÖsszes minta (npy): {len(y_new)}")
print(f"  Train : {(spl_new=='train').sum()}")
print(f"  Val   : {(spl_new=='val').sum()}")
print(f"  Test  : {(spl_new=='test').sum()}")

## Összefoglalás

| | |
|---|---|
| **Augmentációs műveletek** | Forgatás (±18°), fényerő, kontraszt, Gaussian-zaj, blur, perspektíva-warp, vágás+resize |
| **Kombinálás** | Véletlenszerű részhalmazuk, legalább 2 aktív/kép |
| **Célzottság** | Csak a `TARGET` db alatti train-osztályok kapnak augmentált képeket |
| **Val/test érintetlen** | Az egyensúlyozás nem szivárog a kiértékelési splitbe |
| **Kimenet** | `data/augmented/{osztály}/` + `split_manifest_aug.csv` + friss `X_*.npy` |
| **Következő lépés** | Futtasd újra a `11_training_classical_ml.ipynb`-t és a `12_training_cnn.ipynb`-t az új adatokkal |